# Hybrid CLM Prompt Address 001

Hosted-GPU execution for the frozen prompt-scoped address diagnostic. This is a mechanism diagnostic, not a rewrite of CLM Conversion Kill Test 001 or the final Granite Hybrid CLM v0.1 milestone decision.


In [ ]:
from pathlib import Path
import hashlib, json, os, subprocess, sys

ROOT = Path('/kaggle/working/mini-cells')
BRANCH = 'codex/granite-hybrid-clm-v0.1'
if not ROOT.exists():
    subprocess.run(['git', 'clone', 'https://github.com/ArcheLabs/mini-cells.git', str(ROOT)], check=True)
subprocess.run(['git', 'fetch', 'origin', BRANCH], cwd=ROOT, check=True)
subprocess.run(['git', 'checkout', '-B', BRANCH, f'origin/{BRANCH}'], cwd=ROOT, check=True)
print({'branch': subprocess.check_output(['git','branch','--show-current'], cwd=ROOT, text=True).strip(), 'head': subprocess.check_output(['git','rev-parse','HEAD'], cwd=ROOT, text=True).strip()})


In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[lm,dev]'], cwd=ROOT, check=True)
subprocess.run([sys.executable, '-m', 'pytest', '-q', 'tests/test_hybrid_clm.py', 'tests/test_hybrid_clm_artifacts.py', 'tests/test_hybrid_clm_prompt_address_001.py'], cwd=ROOT, check=True)


In [ ]:
try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    os.environ['GITHUB_TOKEN'] = secrets.get_secret('GITHUB_TOKEN')
    os.environ['HF_TOKEN'] = secrets.get_secret('HF_TOKEN')
except Exception as exc:
    raise RuntimeError('Kaggle Secrets GITHUB_TOKEN and HF_TOKEN are both required') from exc
assert os.environ.get('GITHUB_TOKEN')
assert os.environ.get('HF_TOKEN')
print({'github_token_loaded': True, 'hf_token_loaded': True})
subprocess.run([sys.executable, 'scripts/research/hybrid_clm_prompt_address_001/publish.py', '--branch', BRANCH, '--preflight-only'], cwd=ROOT, check=True)


In [ ]:
import torch
protocol_path = ROOT / 'research/validations/hybrid-clm-prompt-address-001/protocol.json'
protocol = json.loads(protocol_path.read_text())
assert protocol['protocol_version'] == 1.0
assert protocol['status'] == 'DIAGNOSTIC_PROTOCOL_FROZEN_GPU_PENDING'
assert protocol['routing']['address_scope'] == 'prompt_anchor'
assert protocol['routing']['candidate_answer_affects_routing'] is False
assert protocol['address_gates']['minimum_heldout_positive_recall'] == 1.0
assert protocol['address_gates']['maximum_history_anchor_false_positive_rate'] == 0.0
assert protocol['write_gates']['maximum_history_kl'] == 0.02
assert protocol['write_gates']['minimum_target_nll_gain'] == 0.5
assert protocol['write_gates']['minimum_semantic_choice_accuracy'] == 1.0
assert protocol['hosted_environment']['require_hf_token'] is True
protocol_sha256 = hashlib.sha256(protocol_path.read_bytes()).hexdigest()
for relative_path, expected in protocol['implementation_git_blobs'].items():
    data = (ROOT / relative_path).read_bytes()
    observed = hashlib.sha1(b'blob ' + str(len(data)).encode() + b'\0' + data).hexdigest()
    assert observed == expected, (relative_path, observed, expected)
assert torch.cuda.is_available()
gpu_info = []
for index in range(torch.cuda.device_count()):
    free, total = torch.cuda.mem_get_info(index)
    gpu_info.append({'index': index, 'name': torch.cuda.get_device_name(index), 'free_mb': free // (1024*1024), 'total_mb': total // (1024*1024)})
print({'protocol_sha256': protocol_sha256, 'seed': protocol['seed'], 'foundation': protocol['base']['model_id'], 'gpus': gpu_info, 'gpu_policy': protocol['gpu_policy']})


In [ ]:
artifact_root = ROOT / 'artifacts/experiments/hybrid-clm-prompt-address-001'
seed = int(protocol['seed'])
durable = artifact_root / f'seed-{seed}/seed_summary.json'
skip = False
if durable.is_file():
    existing = json.loads(durable.read_text())
    if existing.get('status') in {'PASS','FAIL'} and existing.get('protocol_sha256') == protocol_sha256 and existing.get('implementation_git_blobs') == protocol['implementation_git_blobs']:
        print(f'[prompt-address-001][seed={seed}] matching terminal artifact already published; skipping GPU run')
        skip = True


In [ ]:
def run_compact(command, log_path):
    log_path = Path(log_path)
    log_path.parent.mkdir(parents=True, exist_ok=True)
    with log_path.open('w', encoding='utf-8') as handle:
        process = subprocess.Popen(command, cwd=ROOT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=os.environ.copy())
        assert process.stdout is not None
        for line in process.stdout:
            handle.write(line); handle.flush()
            if line.startswith('[granite-hybrid-clm-v0.1]') or line.startswith('{'):
                print(line, end='')
        returncode = process.wait()
    if returncode != 0:
        tail = log_path.read_text(encoding='utf-8', errors='replace').splitlines()[-120:]
        print('\n=== Child log tail ===')
        print('\n'.join(tail))
        subprocess.run(['nvidia-smi'], check=False)
        raise RuntimeError(f'prompt-address child failed with exit code {returncode}; full log: {log_path}')
if not skip:
    run_compact([sys.executable, 'scripts/research/hybrid_clm_prompt_address_001/run.py', '--device', 'cuda:0'], ROOT / f'results/hybrid-clm-prompt-address-001-launcher/seed-{seed}.log')
    subprocess.run([sys.executable, 'scripts/research/hybrid_clm_prompt_address_001/publish.py', '--seed', str(seed), '--branch', BRANCH], cwd=ROOT, check=True)


In [ ]:
decision_path = ROOT / 'artifacts/experiments/hybrid-clm-prompt-address-001/decision.json'
if not decision_path.is_file():
    subprocess.run(['git', 'fetch', 'origin', BRANCH], cwd=ROOT, check=True)
    subprocess.run(['git', 'reset', '--hard', f'origin/{BRANCH}'], cwd=ROOT, check=True)
if decision_path.is_file():
    print(json.dumps(json.loads(decision_path.read_text()), indent=2, sort_keys=True))
else:
    raise RuntimeError('durable decision.json was not found after execution/publish')


Recovery rule: rerun this notebook. A terminal result is skipped only when both the frozen protocol SHA-256 and registered implementation Git-blob map match. Scientific FAIL is published. Do not modify thresholds after GPU observation and call the rerun the same protocol.
